# 📝 Kuis Bab 11 — LLM API & Orkestrasi

10 soal (6 pilihan ganda + 4 coding), total **22 poin**. Penilaian otomatis di cell terakhir.

**Aturan:** kerjakan tanpa membuka materi/lab. Soal coding diisi di bawah `# TODO`,
lalu jalankan cell penilaian. Kuis ini **mandiri** — provider palsunya sudah
disediakan di cell setup (tanpa jaringan, tanpa API key).


## Setup (GIVEN) — Provider Palsu Mini

Provider di bawah ini gagal sesuai daftar `rencana` lalu pulih — cukup untuk menguji
retry tanpa menunggu jam dinding. `TOOLS` berisi dua tool dengan contoh argumen.


In [ ]:
PESAN = [{'role': 'user', 'content': 'Halo'}]


class GalatAPI(Exception):
    def __init__(self, jenis, pesan='gagal'):
        super().__init__(f'[{jenis}] {pesan}')
        self.jenis = jenis


class GalatRateLimit(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('rate_limit', pesan)


class GalatServer(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('server_error', pesan)


class GalatTimeout(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('timeout', pesan)


class GalatPermintaanBuruk(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('bad_request', pesan)


class GalatSemuaPercobaan(GalatAPI):
    def __init__(self, riwayat):
        super().__init__('semua_gagal', 'semua percobaan gagal')
        self.riwayat = list(riwayat)


JENIS_TRANSIEN = ('rate_limit', 'server_error', 'timeout')
GALAT = {'rate_limit': GalatRateLimit, 'server_error': GalatServer,
         'timeout': GalatTimeout, 'bad_request': GalatPermintaanBuruk}


class ProviderUji:
    """Provider palsu: status panggilan ke-n diambil dari `rencana` (habis -> ok)."""

    def __init__(self, rencana=(), latency_ms=200):
        self.rencana = list(rencana)
        self.latency_ms = latency_ms
        self.panggilan = 0

    def panggil(self, messages=None):
        self.panggilan += 1
        status = self.rencana.pop(0) if self.rencana else 'ok'
        if status != 'ok':
            raise GALAT[status]('uji')
        return {'teks': 'ok', 'latency_ms': self.latency_ms, 'attempts': self.panggilan}


TOOLS = {
    'get_cuaca': {'argumen': {'kota': 'Jakarta', 'satuan': 'celsius'}},
    'hitung': {'argumen': {'ekspresi': '1 + 1'}},
}


## Bagian A — Pilihan Ganda (isi `JAWABAN`)


In [ ]:
# Isi jawaban pilihan ganda di sini, contoh: JAWABAN[1] = 'a'

JAWABAN = {
    1: None,
    2: None,
    3: None,
    4: None,
    5: None,
    6: None,
}


### Soal 1 — Galat yang TIDAK boleh diretry

Provider mengembalikan galat "model tidak ada di daftar" (`bad_request`). Manakah
perlakuan yang BENAR?

- a) Retry 3 kali dengan backoff 1-2-4 detik
- b) Retry tanpa batas sampai berhasil
- c) Langsung lempar galat ke pemanggil — percobaan ulang tidak akan mengubah
  apa pun
- d) Retry dengan ganti model besar supaya lebih pintar


### Soal 2 — Kenapa backoff punya batas atas

Backoff eksponensial 1000 → 2000 → 4000 → 8000 ms, lalu dibatasi `maks_ms`.
Alasan utama batas atas itu ada:

- a) Supaya kodenya lebih pendek
- b) Supaya worst-case latency tetap dapat diterima — tanpa batas, percobaan
  ke-10 akan menunggu belasan menit
- c) Karena provider menolak jeda lebih dari 8 detik
- d) Supaya lebih banyak percobaan bisa dicoba dalam waktu yang sama


### Soal 3 — Harga sebuah fallback

Rantai fallback bergerak dari `besar` → `mini` → `murah`. Pernyataan paling tepat:

- a) Ketersediaan naik; yang dibayar adalah konsistensi & kualitas — jadi
  turun-kualitas harus dicatat dan diaudit, bukan disembunyikan
- b) Kualitas jawaban selalu sama; yang berubah hanya biaya
- c) Fallback meniadakan trade-off, jadi aman dipasang di semua jenis tugas
- d) Fallback hanya boleh dipakai untuk tugas kreatif, bukan ekstraksi data


### Soal 4 — Isi kunci cache

Sebuah cache prompt di-key agar permintaan identik tidak dibayar dua kali.
Manakah yang WAJIB masuk kunci?

- a) Cukup teks prompt dari pengguna
- b) Model + pesan + parameter generasi (dan di produksi: versi prompt + konteks
  retrieval)
- c) Cukup nama model dan id pengguna
- d) Cukup timestamp permintaan


### Soal 5 — Batas iterasi pada loop tool

`loop_tool` menerima `maks_iterasi`. Kalau parameter itu dihapus:

- a) Tidak ada yang berubah selama modelnya pintar
- b) Tool akan dipanggil dua kali untuk pertanyaan yang sama
- c) Hasil tool menjadi tidak valid
- d) Satu permintaan bisa berputar tanpa henti — biaya dan latensi jadi tak
  terbatas bila model terus minta tool


### Soal 6 — Streaming: apa yang sebenarnya berubah

Streaming menurunkan TTFT dari 340 ms (total) menjadi 120 ms. Yang SEBENARNYA
berubah:

- a) Total waktu proses menjadi lebih cepat
- b) Biaya token berkurang karena chunk dikirim lebih awal
- c) Waktu sampai pengguna melihat token pertama (persepsi responsif); total
  proses tetap sama
- d) Kualitas jawaban meningkat karena dihasilkan bertahap


## Bagian B — Coding


### Soal 7 — `backoff_ms` (2 poin)

`backoff_ms(percobaan, basis_ms=1000, faktor=2, maks_ms=8000)` → jeda sebelum
percobaan ke-n: `basis_ms * faktor ** (n - 1)`, **dibatasi** `maks_ms`.

Contoh: `[backoff_ms(n) for n in range(1, 6)]` → `[1000, 2000, 4000, 8000, 8000]`.


In [ ]:
def backoff_ms(percobaan, basis_ms=1000, faktor=2, maks_ms=8000):
    # TODO
    pass

# print([backoff_ms(n) for n in range(1, 6)])


### Soal 8 — `panggil_dengan_retry` (5 poin)

`panggil_dengan_retry(provider, messages, maks_percobaan=4, basis_ms=1000,
faktor=2, maks_ms=8000, sleep_fn=None)` → panggil `provider.panggil(messages)`
dengan aturan:

1. Ulangi **hanya** galat transien (`JENIS_TRANSIEN`); galat lain langsung dilempar.
2. Sebelum mengulang, `sleep_fn(jeda)` dengan `jeda = backoff_ms(percobaan, ...)`
   (`sleep_fn=None` → tidak menunggu apa-apa).
3. **Jangan** menunggu setelah percobaan terakhir.
4. Semua percobaan habis → `GalatSemuaPercobaan(riwayat)` dengan
   `riwayat = [(percobaan, jenis), ...]`.

Hasil sukses = dict provider + kunci `attempts`, `timpenungguan_ms`,
`total_ms` (= `latency_ms` + total jeda), `riwayat`.


In [ ]:
def panggil_dengan_retry(provider, messages, maks_percobaan=4, basis_ms=1000,
                         faktor=2, maks_ms=8000, sleep_fn=None):
    # TODO
    pass

# print(panggil_dengan_retry(ProviderUji(['rate_limit', 'ok']), PESAN))


### Soal 9 — `validasi_argumen` (5 poin)

`validasi_argumen(nama, argumen, tools=TOOLS)` → daftar error, format stabil:

- `'missing:<kunci>'` untuk argumen wajib yang hilang (semua kunci contoh = wajib);
- `'type:<kunci>'` bila tipe tidak cocok dengan tipe nilai contoh (`str`, `int,`
  `float`, `bool`);
- `'tak_dikenal:<kunci>'` untuk kunci yang tidak ada di contoh argumen;
- nama tool asing → `['tak_dikenal:<nama>']`; argumen bukan dict → `['type:<argumen>']`.

Urutan error: `missing:` dulu (urut kunci wajib), lalu `tak_dikenal:`/`type:`
urut kunci argumen yang diberikan.


In [ ]:
def validasi_argumen(nama, argumen, tools=TOOLS):
    # TODO
    pass

# print(validasi_argumen('get_cuaca', {}))
# print(validasi_argumen('get_cuaca', {'kota': 5, 'satuan': 'celsius'}))


### Soal 10 — `konsumsi_stream` (4 poin)

`konsumsi_stream(chunks)` merakit chunk `{'delta', 't_ms'}` menjadi:

- `teks` — gabungan semua `delta`;
- `jumlah_chunk` — banyak chunk dengan `delta` **non-kosong**;
- `ttft_ms` — `t_ms` chunk non-kosong PERTAMA (`None` bila tidak ada kata);
- `total_ms` — `t_ms` chunk TERAKHIR (0 bila tidak ada chunk).


In [ ]:
def konsumsi_stream(chunks):
    # TODO
    pass

# print(konsumsi_stream([{'delta': '', 't_ms': 10}, {'delta': 'Halo', 't_ms': 55}]))


## 🧮 Penilaian Otomatis


In [ ]:
total_bobot = 22


def jalankan_kuis():
    skor = 0
    rincian = []

    def cek_pilihan(nomor, kunci, bobot=1):
        nonlocal skor
        benar = JAWABAN.get(nomor) == kunci
        skor += bobot if benar else 0
        rincian.append((nomor, benar))

    def cek_kode(nomor, uji, bobot=1):
        nonlocal skor
        benar = False
        try:
            benar = bool(uji())
        except Exception as e:
            print(f'  soal {nomor}: error -> {type(e).__name__}: {e}')
        skor += bobot if benar else 0
        rincian.append((nomor, benar))

    def _safe(nama, *args, **kwargs):
        fn = globals().get(nama)
        if fn is None:
            return None
        try:
            return fn(*args, **kwargs)
        except Exception:
            return None

    cek_pilihan(1, 'c')
    cek_pilihan(2, 'b')
    cek_pilihan(3, 'a')
    cek_pilihan(4, 'b')
    cek_pilihan(5, 'd')
    cek_pilihan(6, 'c')

    # Soal 7 - backoff_ms
    def uji7():
        assert _safe('backoff_ms', 1) == 1000
        assert [_safe('backoff_ms', n) for n in range(1, 6)] == [1000, 2000, 4000, 8000, 8000]
        assert [_safe('backoff_ms', n, 100, 3, 500) for n in range(1, 5)] == [100, 300, 500, 500]
        assert _safe('backoff_ms', 3, 500, 2, 1500) == 1500
        return True
    cek_kode(7, uji7, bobot=2)

    # Soal 8 - panggil_dengan_retry
    def uji8():
        fn = globals().get('panggil_dengan_retry')
        if fn is None:
            return False
        tunggu = []
        p1 = ProviderUji(['rate_limit', 'ok'])
        h1 = fn(p1, PESAN, sleep_fn=tunggu.append)
        assert h1 is not None and h1['attempts'] == 2, h1
        assert tunggu == [1000], tunggu
        assert h1['timpenungguan_ms'] == [1000]
        assert h1['total_ms'] == h1['latency_ms'] + 1000
        assert h1['riwayat'] == [(1, 'rate_limit')]

        p2 = ProviderUji(['rate_limit', 'server_error'])
        try:
            fn(p2, PESAN, maks_percobaan=2, sleep_fn=lambda ms: None)
        except GalatSemuaPercobaan as e:
            assert e.riwayat == [(1, 'rate_limit'), (2, 'server_error')], e.riwayat
        else:
            raise AssertionError('harus GalatSemuaPercobaan')

        p3 = ProviderUji(['bad_request'])
        try:
            fn(p3, PESAN, sleep_fn=lambda ms: None)
        except GalatPermintaanBuruk:
            pass
        else:
            raise AssertionError('bad_request harus langsung dilempar')
        assert p3.panggilan == 1, p3.panggilan

        tunggu4 = []
        p4 = ProviderUji(['rate_limit'])
        try:
            fn(p4, PESAN, maks_percobaan=1, sleep_fn=tunggu4.append)
        except GalatSemuaPercobaan:
            pass
        else:
            raise AssertionError('maks_percobaan=1 harus GalatSemuaPercobaan')
        assert tunggu4 == [], 'jangan menunggu setelah percobaan terakhir'
        return True
    cek_kode(8, uji8, bobot=5)

    # Soal 9 - validasi_argumen
    def uji9():
        assert _safe('validasi_argumen', 'get_cuaca',
                     {'kota': 'Bandung', 'satuan': 'celsius'}) == []
        assert _safe('validasi_argumen', 'get_cuaca', {}) == ['missing:kota', 'missing:satuan']
        assert _safe('validasi_argumen', 'get_cuaca', {'kota': 5, 'satuan': 'celsius'}) == \
            ['type:kota']
        assert _safe('validasi_argumen', 'get_cuaca', {'satuan': 'celsius'}) == ['missing:kota']
        assert _safe('validasi_argumen', 'get_cuaca',
                     {'kota': 'a', 'satuan': 'c', 'x': 1}) == ['tak_dikenal:x']
        assert _safe('validasi_argumen', 'nuklir', {}) == ['tak_dikenal:nuklir']
        assert _safe('validasi_argumen', 'hitung', {'ekspresi': 3}) == ['type:ekspresi']
        assert _safe('validasi_argumen', 'hitung', {'ekspresi': '1+1'}) == []
        assert _safe('validasi_argumen', 'get_cuaca', ['kota']) == ['type:<argumen>']
        return True
    cek_kode(9, uji9, bobot=5)

    # Soal 10 - konsumsi_stream
    def uji10():
        chunks = [{'delta': '', 't_ms': 10}, {'delta': 'Halo', 't_ms': 55},
                  {'delta': ' dunia', 't_ms': 90}, {'delta': '', 't_ms': 110}]
        s = _safe('konsumsi_stream', chunks)
        assert s is not None, 'konsumsi_stream belum jadi'
        assert s['teks'] == 'Halo dunia', s['teks']
        assert s['jumlah_chunk'] == 2, s['jumlah_chunk']
        assert s['ttft_ms'] == 55, s['ttft_ms']
        assert s['total_ms'] == 110, s['total_ms']
        kosong = _safe('konsumsi_stream', [{'delta': '', 't_ms': 42}])
        assert kosong['ttft_ms'] is None and kosong['teks'] == '' and kosong['jumlah_chunk'] == 0
        nol = _safe('konsumsi_stream', [])
        assert nol['teks'] == '' and nol['total_ms'] == 0
        return True
    cek_kode(10, uji10, bobot=4)

    print('=' * 46)
    for nomor, benar in rincian:
        print(f'  Soal {nomor:>2}: {"BENAR" if benar else "salah"}')
    print('=' * 46)
    print(f'SKOR AKHIR: {skor}/{total_bobot}')
    if skor == total_bobot:
        print('Sempurna! Lanjut ke project starter llmkit.')
    elif skor >= 17:
        print('Bagus! Review soal yang salah, lalu lanjut.')
    else:
        print('Ulangi bagian terkait di lab, coba lagi besok.')


jalankan_kuis()


---

Sudah mencoba serius? Bandingkan pendekatanmu dengan
`03_kunci_jawaban_kuis_llm_api.ipynb` — fokus pada **kenapa**,
bukan sekadar jawaban benar/salah.
